# Ablation: Learning Rate (LR)

Comparing different learning rates:
- **LR=1e-6**: Lower learning rate
- **LR=2e-6**: Medium-low learning rate
- **LR=5e-6 (Default)**: Default learning rate (baseline)
- **LR=1e-5**: Higher learning rate
- **LR=2e-5**: Much higher learning rate

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
from pathlib import Path

# Import shared ablation utilities
from ablation_utils import (
    setup_plotting_style,
    load_all_ablation_models,
    load_all_models_all_metrics,
    make_latex_ablation_table,
    plot_ablation_line,
    plot_ablation_bars,
    compute_deltas,
    print_summary,
    METRICS, METRIC_DISPLAY, METRIC_COLORS
)

# Set up plotting style
setup_plotting_style()

In [3]:
# =============================================================================
# CONFIGURATION - Define ablation models
# =============================================================================

ABLATION_MODELS = {
    "LR=1e-6": {
        "csv_path": "../evaluation/ablations/31-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr1ee-6_wd1e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_lr_1e-6.csv",
        "is_baseline": False,
        "description": "Lower learning rate",
        "lr_value": 1e-6
    },
    "LR=2e-6": {
        "csv_path": "../evaluation/ablations/31-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr2ee-6_wd1e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_lr_2e-6.csv",
        "is_baseline": False,
        "description": "Medium-low learning rate",
        "lr_value": 2e-6
    },
    "CS-CLIP (LR=5e-6)": {
        "csv_path": "../evaluation/exp_csv/19-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd1e-2_neg_rel0.0_inplace1.0_swap1.0_csclip-negclip-hard-new_cleaned.csv",
        "is_baseline": True,
        "description": "Default learning rate (baseline)",
        "lr_value": 5e-6
    },
    "LR=1e-5": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr1ee-5_wd1e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_lr_1e-5.csv",
        "is_baseline": False,
        "description": "Higher learning rate",
        "lr_value": 1e-5
    },
    "LR=2e-5": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr2ee-5_wd1e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_lr_2e-5.csv",
        "is_baseline": False,
        "description": "Much higher learning rate",
        "lr_value": 2e-5
    },
}

# Primary metric for comparison
PRIMARY_METRIC = "text_contrastive_accuracy"

# Checkpoint selection (use best or specific step)
CHECKPOINT_STEP = None  # None = use best checkpoint, or specify step like 5000

# Ablation metadata
ABLATION_NAME = "LEARNING RATE ABLATION"
PARAM_KEY = "lr_value"
PARAM_LABEL = 'Learning Rate'

print("Ablation: Learning Rate (LR)")
print("="*50)
for name, cfg in ABLATION_MODELS.items():
    baseline_mark = " [BASELINE]" if cfg["is_baseline"] else ""
    print(f"  {name}{baseline_mark}: {cfg['description']}")

Ablation: Learning Rate (LR)
  LR=1e-6: Lower learning rate
  LR=2e-6: Medium-low learning rate
  CS-CLIP (LR=5e-6) [BASELINE]: Default learning rate (baseline)
  LR=1e-5: Higher learning rate
  LR=2e-5: Much higher learning rate


In [4]:
# =============================================================================
# LOAD DATA - Single Metric (Primary)
# =============================================================================

scores_df = load_all_ablation_models(ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP)
print(f"\nLoaded {len(scores_df)} models, {len(scores_df.columns)} datasets")

Loading LR=1e-6...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded 65 datasets (step=15000)
Loading LR=2e-6...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded 65 datasets (step=20000)
Loading CS-CLIP (LR=5e-6)...
[apply_mappings] Dropped 16 original rows replaced by aliased metrics
  Loaded 65 datasets (step=15000)
Loading LR=1e-5...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded 65 datasets (step=20000)
Loading LR=2e-5...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded 65 datasets (step=20000)

Common datasets (65): ['VG_Relation', 'VG_Attribution', 'ColorFoil', 'VL_CheckList/attr_state', 'VL_CheckList/rel_action', 'VL_CheckList/obj_size', 'VL_CheckList/obj_location', 'VL_CheckList/rel_spatial', 'VL_CheckList/attr_size', 'VL_CheckList/attr_action', 'VL_CheckList/attr_color', 'VL_CheckList/attr_material', 'SPEC/absolute_size', 'VALSE/actions', 'ControlledImages/COCO-

In [5]:
# =============================================================================
# DISPLAY RAW SCORES TABLE
# =============================================================================

# Convert to percentage and display
scores_pct = scores_df * 100

# Add average column
scores_pct['Average'] = scores_pct.mean(axis=1)

print("\n" + "="*60)
print(f"ABLATION: LEARNING RATE")
print(f"Metric: {PRIMARY_METRIC}")
print("="*60)
display(scores_pct.round(1).style.highlight_max(axis=0, color='lightgreen'))


ABLATION: LEARNING RATE
Metric: text_contrastive_accuracy


,VG_Relation,VG_Attribution,ColorFoil,VL_CheckList/attr_state,VL_CheckList/rel_action,VL_CheckList/obj_size,VL_CheckList/obj_location,VL_CheckList/rel_spatial,VL_CheckList/attr_size,VL_CheckList/attr_action,VL_CheckList/attr_color,VL_CheckList/attr_material,SPEC/absolute_size,VALSE/actions,ControlledImages/COCO-One,SugarCrepe/replace_obj,SugarCrepe++/swap_object,MMVP/State,ColorSwap,SugarCrepe/replace_att,ControlledImages/B,MMVP/Spatial,SugarCrepe++/swap_atribute,VALSE/noun phrases,SPEC/existence,SugarCrepe++/replace_object,SugarCrepe/swap_att,ControlledImages/VG-One,VALSE/coreference,SPEC/absolute_spatial,MMVP/Structural Character,ControlledImages/COCO-Two,SugarCrepe/replace_rel,VALSE/relations,NegBench/COCO_val_mcq_llama3.1_rephrased,NegBench/VOC2007_mcq_llama3.1_rephrased,COCO-CF,SugarCrepe++/replace_relation,BLA/co,MMVP/Orientation,VALSE/existence,SPEC/count,NegBench/msr_vtt_mcq_rephrased_llama,ControlledImages/VG-Two,SugarCrepe/swap_obj,SugarCrepe++/replace_attribute,BLA/ap,COLA/multi_objects,MMVP/Color,Flickr30k_Order,ControlledImages/A,SugarCrepe/add_obj,VALSE/counting,SPEC/relative_size,MMVP/Quantity,SPEC/relative_spatial,COCO_Order,Winoground,SugarCrepe/add_att,BLA/rc,VALSE/plurals,MMVP/Presence,MMVP/Camera Perspective,MMVP/Text,VisMin,Average
Model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
LR=1e-6,85.000000,67.900000,88.800000,64.800000,83.500000,92.800000,93.800000,79.800000,61.100000,77.600000,78.300000,71.800000,42.300000,83.300000,47.400000,94.700000,51.400000,13.300000,56.300000,88.100000,34.100000,40.000000,52.400000,92.900000,63.300000,89.900000,72.500000,44.100000,56.400000,12.300000,13.300000,50.000000,78.000000,72.100000,35.100000,39.000000,78.500000,58.800000,48.500000,6.700000,80.000000,34.100000,31.500000,52.300000,67.800000,72.000000,50.400000,35.700000,0.000000,95.000000,30.600000,87.800000,66.700000,32.500000,13.300000,29.900000,93.800000,27.500000,78.600000,48.400000,70.300000,6.700000,20.000000,0.000000,78.600000,56.400000
LR=2e-6,83.100000,68.000000,88.400000,65.500000,83.200000,93.100000,94.000000,78.300000,61.300000,78.100000,78.600000,73.100000,40.100000,83.000000,50.600000,95.100000,51.800000,13.300000,59.000000,86.900000,35.500000,26.700000,56.600000,93.100000,66.300000,91.000000,74.600000,43.500000,55.900000,12.200000,13.300000,50.500000,79.400000,69.700000,33.000000,35.400000,78.600000,60.800000,49.200000,20.000000,80.400000,34.700000,31.500000,53.900000,70.600000,72.300000,50.700000,37.600000,6.700000,96.200000,31.100000,89.700000,67.600000,30.900000,0.000000,30.000000,94.800000,30.800000,80.500000,49.300000,71.700000,0.000000,26.700000,0.000000,78.700000,56.700000
CS-CLIP (LR=5e-6),86.300000,69.400000,90.500000,65.200000,83.000000,93.100000,94.100000,80.700000,64.300000,78.100000,79.200000,74.500000,38.800000,83.200000,48.000000,94.900000,52.200000,13.300000,59.000000,86.200000,34.600000,46.700000,56.500000,93.700000,68.700000,91.700000,74.500000,45.100000,56.200000,12.200000,6.700000,50.900000,79.700000,70.100000,31.400000,37.800000,78.200000,62.400000,48.400000,13.300000,83.400000,34.500000,29.200000,53.100000,69.000000,74.200000,52.700000,41.000000,13.300000,96.700000,29.100000,90.800000,66.500000,32.700000,6.700000,29.500000,95.300000,29.800000,80.800000,49.600000,70.500000,0.000000,6.700000,13.300000,78.600000,57.200000
LR=1e-5,85.700000,68.600000,90.300000,64.000000,83.000000,93.200000,94.100000,81.500000,63.700000,78.000000,79.300000,75.300000,39.400000,83.300000,43.900000,94.200000,53.100000,6.700000,58.300000,88.300000,34.800000,33.300000,59.800000,93.800000,69.500000,90.100000,76.600000,47.100000,60.300000,13.100000,26.700000,49.800000,80.800000,71.600000,29.000000,35.800000,76.800000,63.900000,48.100000,6.700000,81.800000,34.600000,31.100000,53.100000,71.400000,74.900000,53.400000,39.000000,6.700000,96.600000,30.600000,90.600000,66.400000,30.700000,13.300000,29.700000,95.300000,28.500000,83.100000,50.000000,70.700000,13.300000,13.300000,6.700000,77.800000

In [6]:
# =============================================================================
# LOAD ALL METRICS (Text, Image, Group Contrastive Accuracy)
# =============================================================================

# Load all models with all metrics
all_metrics_df = load_all_models_all_metrics(ABLATION_MODELS, METRICS, CHECKPOINT_STEP)

# Extract just the summary columns (I2T, T2I, Group)
summary_cols = [col for col in ['I2T', 'T2I', 'Group'] if col in all_metrics_df.columns]
summary_df = all_metrics_df[summary_cols].copy()

# Add overall average
summary_df['Average'] = summary_df.mean(axis=1)

print("\n" + "="*60)
print("ABLATION: LEARNING RATE - ALL METRICS")
print("="*60)
display((summary_df * 100).round(1).style.highlight_max(axis=0, color='lightgreen'))

Loading LR=1e-6...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']
Loading LR=2e-6...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']
Loading CS-CLIP (LR=5e-6)...
[apply_mappings] Dropped 16 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']
Loading LR=1e-5...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']
Loading LR=2e-5...
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
  Loaded metrics: ['I2T', 'T2I', 'Group']

Common datasets across all models (19): ['BLA', 'COCO-CF', 'COCO_Order', 'COLA', 'ColorFoil', 'ColorSwap', 'ControlledImages', 'Flickr30k_Order', 'MMVP', 'NegBench', 'SPEC', 'SugarCrepe', 'SugarCrepe++', 'VALSE', 'VG_Attribution', 'VG_Relation', 'VL_CheckList', 'VisMin', 'Winoground']

ABLATION: LEARNING RATE - ALL METRICS


,I2T,T2I,Group,Average
Model,,,,
LR=1e-6,62.200000,41.700000,24.800000,42.900000
LR=2e-6,62.700000,40.800000,24.800000,42.800000
CS-CLIP (LR=5e-6),63.400000,41.600000,25.400000,43.400000
LR=1e-5,63.100000,42.000000,25.500000,43.600000
LR=2e-5,63.100000,42.500000,25.600000,43.800000


In [7]:
# =============================================================================
# LATEX TABLE GENERATION
# =============================================================================

# Generate LaTeX table
latex_table = make_latex_ablation_table(
    summary_df,
    ABLATION_MODELS,
    caption="Learning rate ablation. I2T = Image-to-Text, T2I = Text-to-Image, Group = both correct. Best in \\textbf{bold}, baseline \\underline{underlined}.",
    label="tab:ablation_learning_rate",
)

print("="*60)
print("LATEX TABLE")
print("="*60)
print(latex_table)

LATEX TABLE
\begin{table}[t]
  \centering
  \small
  \caption{Learning rate ablation. I2T = Image-to-Text, T2I = Text-to-Image, Group = both correct. Best in \textbf{bold}, baseline \underline{underlined}.}
  \label{tab:ablation_learning_rate}
  \begin{tabular}{lcccc}
    \toprule
    Model & I2T & T2I & Group & Average \\
    \midrule
    LR=1e-6 & 62.2 & 41.7 & 24.8 & 42.9 \\
    LR=2e-6 & 62.7 & 40.8 & 24.8 & 42.8 \\
    CS-CLIP (LR=5e-6) & \textbf{\underline{63.4}} & \underline{41.6} & \underline{25.4} & \underline{43.4} \\
    LR=1e-5 & 63.1 & 42.0 & 25.5 & 43.6 \\
    LR=2e-5 & 63.1 & \textbf{42.5} & \textbf{25.6} & \textbf{43.8} \\
    \bottomrule
  \end{tabular}
\end{table}


In [9]:
# =============================================================================
# COMPUTE DELTAS FROM BASELINE
# =============================================================================

deltas_df = compute_deltas(summary_df, ABLATION_MODELS)

print("\n" + "="*60)
print("DELTA FROM BASELINE (percentage points)")
print("="*60)
display(deltas_df.round(2).style.background_gradient(cmap='RdYlGn', axis=None))


DELTA FROM BASELINE (percentage points)


,I2T,T2I,Group,Average
Model,,,,
LR=1e-6,-1.200000,0.110000,-0.610000,-0.570000
LR=2e-6,-0.670000,-0.770000,-0.620000,-0.690000
CS-CLIP (LR=5e-6),0.000000,0.000000,0.000000,0.000000
LR=1e-5,-0.260000,0.490000,0.120000,0.120000
LR=2e-5,-0.240000,1.000000,0.220000,0.330000


In [10]:
# =============================================================================
# SUMMARY
# =============================================================================

print_summary(summary_df, ABLATION_MODELS, ABLATION_NAME, PARAM_KEY)


SUMMARY: LEARNING RATE ABLATION

Baseline: CS-CLIP (LR=5e-6)

Average Performance:
    LR=1e-6: 42.9% (-0.57pp vs baseline) | lr_value=1e-06
    LR=2e-6: 42.8% (-0.69pp vs baseline) | lr_value=2e-06
  ★ CS-CLIP (LR=5e-6): 43.4% (+0.00pp vs baseline) | lr_value=5e-06
    LR=1e-5: 43.6% (+0.12pp vs baseline) | lr_value=1e-05
    LR=2e-5: 43.8% (+0.33pp vs baseline) | lr_value=2e-05

Key Findings:
  - Best: LR=2e-5 (43.8%)
  - Worst: LR=2e-6 (42.8%)
  - Gap: 1.0pp


In [11]:
# =============================================================================
# DATASET-WISE AND SUBSET-WISE TABLES (with ARO merging)
# =============================================================================

from ablation_utils import (
    load_all_models_per_dataset,
    load_all_models_per_subset,
    make_latex_dataset_table,
    get_datasets_and_subsets,
    display_all_tables,
    load_benchmark_config
)

# Load benchmark config for dataset merge rules (e.g., ARO)
bench_cfg = load_benchmark_config()

# Display all tables for the primary metric (I2T) with ARO merging
dataset_df, subset_df, datasets_subsets = display_all_tables(
    ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP, 
    show_latex=True, apply_merge=True, benchmark_config=bench_cfg
)


PER-DATASET RESULTS (I2T)
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
[apply_mappings] Dropped 16 original rows replaced by aliased metrics
[apply_mappings] Dropped 8 original rows replaced by aliased metrics
[apply_mappings] Dropped 8 original rows replaced by aliased metrics


,ARO,BLA,COCO-CF,COLA,ColorFoil,ColorSwap,ControlledImages,MMVP,NegBench,SPEC,SugarCrepe,SugarCrepe++,VALSE,VL_CheckList,VisMin,Winoground,Average
Model,,,,,,,,,,,,,,,,,
LR=1e-6,85.400000,49.100000,78.500000,35.700000,88.800000,56.300000,43.100000,12.600000,35.200000,35.700000,81.100000,64.900000,74.500000,78.200000,78.600000,27.500000,57.800000
LR=2e-6,85.500000,49.700000,78.600000,37.600000,88.400000,59.000000,44.200000,11.900000,33.300000,35.700000,82.400000,66.500000,74.500000,78.400000,78.700000,30.800000,58.400000
CS-CLIP (LR=5e-6),86.900000,50.200000,78.200000,41.000000,90.500000,59.000000,43.500000,13.300000,32.800000,36.100000,82.200000,67.400000,74.800000,79.200000,78.600000,29.800000,59.000000
LR=1e-5,86.600000,50.500000,76.800000,39.000000,90.300000,58.300000,43.200000,14.100000,32.000000,36.200000,83.600000,68.300000,75.400000,79.100000,77.800000,28.500000,58.700000
LR=2e-5,86.700000,50.500000,75.500000,37.600000,90.400000,63.300000,42.300000,15.600000,30.700000,35.900000,83.600000,67.300000,75.700000,79.300000,77.700000,27.800000,58.700000



LaTeX:
\begin{table}[t]
  \centering
  \scriptsize
  \caption{Per-dataset I2T accuracy.}
  \label{tab:ablation_datasets_i2t}
  \begin{adjustbox}{max width=\textwidth}
  \begin{tabular}{lccccccccccccccccc}
    \toprule
    Model & \rotatebox{60}{ARO} & \rotatebox{60}{BLA} & \rotatebox{60}{COCO-CF} & \rotatebox{60}{COLA} & \rotatebox{60}{ColorFoil} & \rotatebox{60}{ColorSwap} & \rotatebox{60}{ControlledImages} & \rotatebox{60}{MMVP} & \rotatebox{60}{NegBench} & \rotatebox{60}{SPEC} & \rotatebox{60}{SugarCrepe} & \rotatebox{60}{SugarCrepe++} & \rotatebox{60}{VALSE} & \rotatebox{60}{VL\_CheckList} & \rotatebox{60}{VisMin} & \rotatebox{60}{Winoground} & \rotatebox{60}{Avg} \\
    \midrule
    LR=1e-6 & 85.4 & 49.1 & 78.5 & 35.7 & 88.8 & 56.3 & 43.1 & 12.6 & \textbf{35.2} & 35.7 & 81.1 & 64.9 & 74.5 & 78.2 & 78.6 & 27.5 & 57.8 \\
    LR=2e-6 & 85.5 & 49.7 & \textbf{78.6} & 37.6 & 88.4 & 59.0 & \textbf{44.2} & 11.9 & 33.3 & 35.7 & 82.4 & 66.5 & 74.5 & 78.4 & \textbf{78.7} & \textbf{30.8} & 5

,ARO/VG_Relation,ARO/VG_Attribution,ColorFoil,VL_CheckList/attr_state,VL_CheckList/rel_action,VL_CheckList/obj_size,VL_CheckList/obj_location,VL_CheckList/rel_spatial,VL_CheckList/attr_size,VL_CheckList/attr_action,VL_CheckList/attr_color,VL_CheckList/attr_material,SPEC/absolute_size,VALSE/actions,ControlledImages/COCO-One,SugarCrepe/replace_obj,SugarCrepe++/swap_object,MMVP/State,ColorSwap,SugarCrepe/replace_att,ControlledImages/B,MMVP/Spatial,SugarCrepe++/swap_atribute,VALSE/noun phrases,SPEC/existence,SugarCrepe++/replace_object,SugarCrepe/swap_att,ControlledImages/VG-One,VALSE/coreference,SPEC/absolute_spatial,MMVP/Structural Character,ControlledImages/COCO-Two,SugarCrepe/replace_rel,VALSE/relations,NegBench/COCO_val_mcq_llama3.1_rephrased,NegBench/VOC2007_mcq_llama3.1_rephrased,COCO-CF,SugarCrepe++/replace_relation,BLA/co,MMVP/Orientation,VALSE/existence,SPEC/count,NegBench/msr_vtt_mcq_rephrased_llama,ControlledImages/VG-Two,SugarCrepe/swap_obj,SugarCrepe++/replace_attribute,BLA/ap,COLA/multi_objects,MMVP/Color,ARO/Flickr30k_Order,ControlledImages/A,SugarCrepe/add_obj,VALSE/counting,SPEC/relative_size,MMVP/Quantity,SPEC/relative_spatial,ARO/COCO_Order,Winoground,SugarCrepe/add_att,BLA/rc,VALSE/plurals,MMVP/Presence,MMVP/Camera Perspective,MMVP/Text,VisMin,Average
Model,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
LR=1e-6,85.000000,67.900000,88.800000,64.800000,83.500000,92.800000,93.800000,79.800000,61.100000,77.600000,78.300000,71.800000,42.300000,83.300000,47.400000,94.700000,51.400000,13.300000,56.300000,88.100000,34.100000,40.000000,52.400000,92.900000,63.300000,89.900000,72.500000,44.100000,56.400000,12.300000,13.300000,50.000000,78.000000,72.100000,35.100000,39.000000,78.500000,58.800000,48.500000,6.700000,80.000000,34.100000,31.500000,52.300000,67.800000,72.000000,50.400000,35.700000,0.000000,95.000000,30.600000,87.800000,66.700000,32.500000,13.300000,29.900000,93.800000,27.500000,78.600000,48.400000,70.300000,6.700000,20.000000,0.000000,78.600000,56.400000
LR=2e-6,83.100000,68.000000,88.400000,65.500000,83.200000,93.100000,94.000000,78.300000,61.300000,78.100000,78.600000,73.100000,40.100000,83.000000,50.600000,95.100000,51.800000,13.300000,59.000000,86.900000,35.500000,26.700000,56.600000,93.100000,66.300000,91.000000,74.600000,43.500000,55.900000,12.200000,13.300000,50.500000,79.400000,69.700000,33.000000,35.400000,78.600000,60.800000,49.200000,20.000000,80.400000,34.700000,31.500000,53.900000,70.600000,72.300000,50.700000,37.600000,6.700000,96.200000,31.100000,89.700000,67.600000,30.900000,0.000000,30.000000,94.800000,30.800000,80.500000,49.300000,71.700000,0.000000,26.700000,0.000000,78.700000,56.700000
CS-CLIP (LR=5e-6),86.300000,69.400000,90.500000,65.200000,83.000000,93.100000,94.100000,80.700000,64.300000,78.100000,79.200000,74.500000,38.800000,83.200000,48.000000,94.900000,52.200000,13.300000,59.000000,86.200000,34.600000,46.700000,56.500000,93.700000,68.700000,91.700000,74.500000,45.100000,56.200000,12.200000,6.700000,50.900000,79.700000,70.100000,31.400000,37.800000,78.200000,62.400000,48.400000,13.300000,83.400000,34.500000,29.200000,53.100000,69.000000,74.200000,52.700000,41.000000,13.300000,96.700000,29.100000,90.800000,66.500000,32.700000,6.700000,29.500000,95.300000,29.800000,80.800000,49.600000,70.500000,0.000000,6.700000,13.300000,78.600000,57.200000
LR=1e-5,85.700000,68.600000,90.300000,64.000000,83.000000,93.200000,94.100000,81.500000,63.700000,78.000000,79.300000,75.300000,39.400000,83.300000,43.900000,94.200000,53.100000,6.700000,58.300000,88.300000,34.800000,33.300000,59.800000,93.800000,69.500000,90.100000,76.600000,47.100000,60.300000,13.100000,26.700000,49.800000,80.800000,71.600000,29.000000,35.800000,76.800000,63.900000,48.100000,6.700000,81.800000,34.600000,31.100000,53.100000,71.400000,74.900000,53.400000,39.000000,6.700000,96.600000,30.600000,90.600000,66.400000,30.700000,13.300000,29.700000,95.300000,28.500000,83.100000,50.000000,70.700000,13.300000,13.300000,6.


LaTeX:
\begin{table}[t]
  \centering
  \scriptsize
  \caption{Per-subset I2T accuracy.}
  \label{tab:ablation_subsets_i2t}
  \begin{adjustbox}{max width=\textwidth}
  \begin{tabular}{lcccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccccc}
    \toprule
    Model & \rotatebox{60}{ARO/VG\_Relation} & \rotatebox{60}{ARO/VG\_Attribution} & \rotatebox{60}{ColorFoil} & \rotatebox{60}{VL\_CheckList/attr\_state} & \rotatebox{60}{VL\_CheckList/rel\_action} & \rotatebox{60}{VL\_CheckList/obj\_size} & \rotatebox{60}{VL\_CheckList/obj\_location} & \rotatebox{60}{VL\_CheckList/rel\_spatial} & \rotatebox{60}{VL\_CheckList/attr\_size} & \rotatebox{60}{VL\_CheckList/attr\_action} & \rotatebox{60}{VL\_CheckList/attr\_color} & \rotatebox{60}{VL\_CheckList/attr\_material} & \rotatebox{60}{SPEC/absolute\_size} & \rotatebox{60}{VALSE/actions} & \rotatebox{60}{ControlledImages/COCO-One} & \rotatebox{60}{SugarCrepe/replace\_obj} & \rotatebox{60}{SugarCrepe++/swap\_object} & \rotatebox{60}{MMVP/S